# **PIPELINE IMPLEMENTATION**

What's a **pipeline**?

A **pipeline** is a way to chain multiple data preprocessing steps and a model together into a single object, so that data flows sequentially through each step — the output of one step becomes the input of the next. It ensures that preprocessing (like scaling, imputing, or encoding) is properly fit only on training data and consistently applied to test data, preventing data leakage. 

It also makes the code cleaner, more reusable, and easier to deploy, since the entire workflow can be treated and saved as a single object

Today, in this notebook our main agenda is to implement pipeline on data of titanic. doing some preprocessing and finally building a decision tree classifier.

In [77]:
# making our imports
import pandas as pd
import numpy as np

# sklearn imports
from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# setting configuration to show diagrams
set_config(display='diagram')

Let's read our dataset and see what it have.

In [78]:
# Reading our data from csv and storing it in variable
df = pd.read_csv("/kaggle/input/datasets/abhinavralhan/titanic/train.csv")

# taking 10 samples out of data to see the content of data and decide further steps
df.sample(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
782,783,0,1,"Long, Mr. Milton Clyde",male,29.0,0,0,113501,30.0000,D6,S
639,640,0,3,"Thorneycroft, Mr. Percival",male,NaN,1,0,376564,16.1000,NaN,S
826,827,0,3,"Lam, Mr. Len",male,NaN,0,0,1601,56.4958,NaN,S
337,338,1,1,"Burns, Miss. Elizabeth Margaret",female,41.0,0,0,16966,134.5000,E40,C
404,405,0,3,"Oreskovic, Miss. Marija",female,20.0,0,0,315096,8.6625,NaN,S
701,702,1,1,"Silverthorne, Mr. Spencer Victor",male,35.0,0,0,PC 17475,26.2875,E24,S
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
212,213,0,3,"Perkin, Mr. John Henry",male,22.0,0,0,A/5 21174,7.2500,NaN,S
665,666,0,2,"Hickman, Mr. Lewis",male,32.0,2,0,S.O.C. 14879,73.5000,NaN,S
438,439,0,1,"Fortune, Mr. Mark",male,64.0,1,4,19950,263.0000,C23 C25 C27,S


In [79]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [80]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


From the above visual of data we can see our data have mix of categorical and numeric columns. let's decide what we gonna do here.

## Planning

* **Step1** : Removing passengerid, name, ticket and cabin as these won't impact our model
* **Step2** : Splitting our data
* **Step3** : Preprocessing data using ColumnTransformer
* **Step4** : Applying MinMaxScaler on the dataset to squeeze data into common range
* **Step5** : Creating an object of DesicionTreeClassifier
* **Step6** : Creating Pipeline

Let's start our work...

## STEP 1 : Removing extra columns 

Removing PassengerId, Name, Ticket, Cabin

In [81]:
# Step 1 : Removing Passengerid, Name, Ticket and Cabin from our data
cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df.drop(columns=cols, inplace=True) # inplace = True updates your original data and return none. so, there's no need to reassign.

# taking a peek at our data after dropping
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## STEP 2 : Splitting data

In this step, we will split our data into X, y (train and test).

In [82]:
# Seperating X_data and y_data
X_data = df.drop('Survived', axis=1)
y_data = df['Survived']

# Splitting our data
X_train, X_test, y_train, y_test = train_test_split(
    X_data,
    y_data,
    random_state = 0,
    train_size = 0.8,
    test_size = 0.2
)

X_data.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


## STEP 3 : Performing preprocessing using column transformer

In this step we'll perform imputation and encoding on our data in same column transformer

In [83]:
# Using Column Transformer to perform imputation
preprocessing = ColumnTransformer(
    [
        ('AgeImputer', SimpleImputer(), ['Age']),
        ('embark_pipe', Pipeline([
            ('EmbarkedImputer', SimpleImputer(strategy='most_frequent')),
            ('Embarked_Encoder', OneHotEncoder(sparse_output=False, dtype=np.int32, handle_unknown='ignore'))
        ]),
        ['Embarked']
        ),
    ('sex_encoder', OneHotEncoder(sparse_output=False, dtype=np.int32, handle_unknown='ignore'), ['Sex'])
    ],
    remainder="passthrough"
)

## STEP 4 : Applying Normalization

In this step, we'll apply MinMaxScaler on our data. the purpose of this scaling is to center the data into same range.

In [84]:
# Using column Transformer to perform normalizations
min_max_scaler_transformer = ColumnTransformer(
    [('Scaler', MinMaxScaler(), slice(1,10))],
    remainder='passthrough'
)

In [85]:
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
140,3,female,NaN,0,2,15.2458,C
439,2,male,31.0,0,0,10.5000,S
817,2,male,31.0,1,1,37.0042,C
378,3,male,20.0,0,0,4.0125,C
491,3,male,21.0,0,0,7.2500,S
...,...,...,...,...,...,...,...
835,1,female,39.0,1,1,83.1583,C
192,3,female,19.0,1,0,7.8542,S
629,3,male,NaN,0,0,7.7333,Q
559,3,female,36.0,1,0,17.4000,S


## STEP 5 : Creating object of DecisionTreeClassifier

In this step, we'll create the object for our DecisionTreeClassifier to put it into our Pipeline.

In [86]:
# Creating object of DecisionTreeClassifier
classifier = DecisionTreeClassifier()

## STEP 6 : CREATING OUR PIPELINE

Now, let's create our pipeline by chaining all the above steps into one pipeline.

In [87]:
# Creating pipeline
pipeline = Pipeline([
    ('Preprocessing', preprocessing),
    ('Normalizer', min_max_scaler_transformer),
    ('Classifier', classifier)
])

In [88]:
pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('Preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('AgeImputer', SimpleImputer(),
                                                  ['Age']),
                                                 ('embark_pipe',
                                                  Pipeline(steps=[('EmbarkedImputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('Embarked_Encoder',
                                                                   OneHotEncoder(dtype=<class 'numpy.int32'>,
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked']),
                                                 ('sex_encoder',
                                                  OneHotEncoder(dtype=<class 'numpy.int32'>,
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Sex'])])),
                ('Normalizer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Scaler', MinMaxScaler(),
                                                  slice(1, 10, None))])),
                ('Classifier', DecisionTreeClassifier())])

## STEP 7 : GENERATING PREDICTIONS AND CALCULATING ACCURACY

In [122]:
# Generating predicitons
y_pred = pipeline.predict(X_test)

# Calculating Accuracy
acc = accuracy_score(y_pred, y_test)

# Printing Accuracy
print(acc)

0.7653631284916201


## CONCLUSION